# Week 6, Day 2: Batch-rewrite memorabilia listings with Groq

**Lesson:** [week6/day2.ipynb](../../../week6/day2.ipynb), *Data pre-processing*. **About 30 minutes.**

**What you practice here**

- A system prompt that has a model rewrite messy text into a fixed format of your own design, as the lesson's five-line product summary does.
- One LiteLLM `completion` call with a low reasoning effort: the reply's text comes from `choices[0].message.content`, and what the call cost from `_hidden_params["response_cost"]`.
- The batch API: a JSONL file of request lines, uploaded with `files.create` and run with `batches.create`, through Groq's client or OpenAI's, which take the same calls.
- Collecting a finished batch with `batches.retrieve` and `files.content`, and matching each reply to its listing by `custom_id`.

**How this notebook works.** Run the first code cell once. Then, for each part below, write it in its cell, run that cell, and run the check under it. The checks hand your code stand-ins shaped like the real client and libraries: they record what your code sends and answer with canned replies, so nothing is called and nothing is spent. Write from memory, and look back at the lesson only when you are stuck.

When every check passes, save the notebook, run the checker and push, as in every session. The tracker then marks the lesson done.

In [ ]:
# Run this cell first. It finds plan/check.py in your fork and loads this exercise's checks.
import os, pathlib, sys
ROOT = next((p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "plan" / "check.py").is_file()), None)
if ROOT is None and ("google.colab" in sys.modules or "COLAB_RELEASE_TAG" in os.environ):
    # in Colab: fetch the checker from your fork on GitHub, and the packages Colab lacks
    import subprocess, urllib.request
    ROOT = pathlib.Path("/content/fork")
    (ROOT / "plan").mkdir(parents=True, exist_ok=True)
    urllib.request.urlretrieve("https://raw.githubusercontent.com/starlest/llm_engineering/main/plan/check.py", ROOT / "plan" / "check.py")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "openai", "python-dotenv", "tiktoken"], check=False)
    print("In Colab, the checks run here too. To keep your work, use File > Save a copy in GitHub, to this notebook's path in your fork.")
if ROOT:
    sys.path.insert(0, str(ROOT / "plan"))
    try:
        from check import exercise
    except ImportError:
        raise ImportError("your plan/check.py is older than this notebook: run git pull, or copy the new one from the tracker page") from None
    check = exercise("week6/day2", globals())
else:  # outside your fork: write and try the parts here, and run the checks in your codespace
    def check(part=None):
        print("The checks are off here, as plan/check.py is not in a folder above this notebook. Run them in your codespace.")
    print("plan/check.py was not found above this notebook, so the checks are off here.")

## The task

The lesson has a model rewrite each product's messy text into five short lines. It first makes one LiteLLM call to
see the result and what it costs, then rewrites every product at once through Groq's batch API. Do the same for
wrestling memorabilia listings, whose sellers write things like "WM III PROGRAM!!! 1987 silverdome, light creases,
lot #A77": design the summary's format, rewrite one listing, then write, send and collect a batch.

The functions that call a model or an API take it as their first parameter. `rewrite` takes LiteLLM's `completion`
function. `submit_batch` and `collect` take a batch client: Groq's `Groq(api_key=...)`, as in the lesson, or
OpenAI's `OpenAI()`, which takes the same batch calls. The checks pass stand-ins shaped like them: they record what
you send and answer with canned replies and files, so nothing is spent.

## What each call should give

```python
rewrite(completion, "WM III PROGRAM!!! 1987 silverdome, light creases, lot #A77")
# makes one request with the model groq/openai/gpt-oss-20b, a low reasoning effort, and two messages: your
# system prompt, then the listing. Returns the summary and the call's cost in dollars:
# ('Title: WrestleMania III official program\nCategory: Programs\nEra: ...', 0.000123)

request_line(7, "royal rumble 1992 arena poster, rolled")
# '{"custom_id": "7", "method": "POST", "url": "/v1/chat/completions", "body": {"model": "openai/gpt-oss-20b", ...}}'

batch_id = submit_batch(groq, "listings.jsonl")
# uploads the file, starts a batch on it, and returns the batch's id: 'batch_01k...'

collect(groq, batch_id, "results.jsonl")
# None while the batch runs; once it is completed: {0: 'Title: ...', 1: 'Title: ...', 2: 'Title: ...'}
```

In [ ]:
# The imports for this exercise. Add more here if you need them: the checks have the standard library, openai, pydantic, dotenv, numpy, bs4 and tiktoken.
import json

### 1 · `SYSTEM_PROMPT`

Your system prompt for rewriting one listing into a standard summary. Like the lesson's, it asks for the summary
alone, in a fixed format: here, five lines that start with the labels `Title:`, `Category:`, `Era:`, `Condition:`
and `Description:`. Tell the model to leave out prices, since the pricers of the coming days must not read the
answer, and lot numbers, as the lesson leaves out part numbers. The wording is yours.

<details><summary>Hint</summary>

The lesson's `SYSTEM_PROMPT` is a triple-quoted string: the task and the rules first, then the format, one label per
line, each followed by what goes there.

```python
SYSTEM_PROMPT = """Rewrite ... Respond only in this format, with no prices and no lot numbers:
Title: ...
Category: ...
Era: ...
Condition: ...
Description: ..."""
```

</details>

In [ ]:
SYSTEM_PROMPT = ""

In [ ]:
check("SYSTEM_PROMPT")

### 2 · `rewrite(completion, text, model="groq/openai/gpt-oss-20b")`

Rewrites one listing with one LiteLLM call and returns the summary with what it cost. `completion` is LiteLLM's
function. Call it once, passing everything by name: the model `model`, a low reasoning effort, as the lesson's first
rewrite does, and two messages: a system message holding `SYSTEM_PROMPT`, then a user message that holds `text`,
unchanged. Return a tuple `(summary, cost)`: the reply's text, and the cost in dollars that LiteLLM adds to the
response.

<details><summary>Hint</summary>

```python
response = completion(model=model, messages=messages, reasoning_effort="low")
return response.choices[0].message.content, response._hidden_params["response_cost"]
```

</details>

In [ ]:
def rewrite(completion, text, model="groq/openai/gpt-oss-20b"):
    # Write rewrite here, as described above.
    raise NotImplementedError("rewrite is not written yet")

In [ ]:
check("rewrite")

### 3 · `request_line(custom_id, text, model="openai/gpt-oss-20b")`

Builds one line of the batch file: the request to rewrite one listing, as the lesson's `make_jsonl` does. Return the
JSON text, on one line, of a dict with four keys: `"custom_id"` (`custom_id` as text, such as `"7"`), `"method"`
(`"POST"`), `"url"` (the Chat Completions path, `"/v1/chat/completions"`) and `"body"`. The body is the request
itself, with exactly three keys: `"model"`, `"messages"` (the same two messages as in `rewrite`) and
`"reasoning_effort"` (`"low"`). The default model is Groq's own name for the model, without LiteLLM's `groq/` in
front. For OpenAI's batch API, pass an OpenAI model, such as `"gpt-5-nano"`.

<details><summary>Hint</summary>

```python
body = {"model": model, "messages": messages, "reasoning_effort": "low"}
return json.dumps({"custom_id": str(custom_id), "method": "POST", "url": "/v1/chat/completions", "body": body})
```

</details>

In [ ]:
def request_line(custom_id, text, model="openai/gpt-oss-20b"):
    # Write request_line here, as described above.
    raise NotImplementedError("request_line is not written yet")

In [ ]:
check("request_line")

### 4 · `submit_batch(client, path)`

Uploads a batch file and starts the batch, as the lesson does in two calls, and returns the batch's id. `client` is
a Groq client or an `OpenAI()` client: both take the same calls. Open the file in binary mode, with no encoding (the
lesson's `encoding="utf-8"` there raises `ValueError`), and upload it with the client's file upload, for the purpose
`"batch"`. Then create a batch on the uploaded file's id, for the endpoint `"/v1/chat/completions"`, with a
completion window of `"24h"`. Return the id of the batch it gives back.

<details><summary>Hint</summary>

```python
with open(path, "rb") as f:
    upload = client.files.create(file=f, purpose="batch")
batch = client.batches.create(completion_window="24h", endpoint="/v1/chat/completions", input_file_id=upload.id)
return batch.id
```

</details>

In [ ]:
def submit_batch(client, path):
    # Write submit_batch here, as described above.
    raise NotImplementedError("submit_batch is not written yet")

In [ ]:
check("submit_batch")

### 5 · `collect(client, batch_id, path)`

Fetches a batch's results once it is done, as a dict from each listing's id to its summary. Look the batch up by its
id, and return `None` while its status is not `"completed"`. Otherwise download its output file's content and save
it at `path` with the response's `write_to_file`, as the lesson does, then read that file. Each line that is not
blank holds one result as JSON: map its `"custom_id"`, turned back into an int, to the reply's text, which sits
inside the result's `"response"` body where a Chat Completions response keeps it.

<details><summary>Hint</summary>

```python
batch = client.batches.retrieve(batch_id)
client.files.content(batch.output_file_id).write_to_file(path)
row = json.loads(line)
row["response"]["body"]["choices"][0]["message"]["content"]
```

</details>

In [ ]:
def collect(client, batch_id, path):
    # Write collect here, as described above.
    raise NotImplementedError("collect is not written yet")

In [ ]:
check("collect")

## Check everything

The cell below runs all 8 checks again. When they all pass, save this notebook (Ctrl+S, or Cmd+S on a Mac), then run the checker and push, as in every session:

```bash
uv run python plan/check.py
git add -A
git commit -m "Week 6, Day 2 exercise"
git push
```

The checker runs the same checks on the saved notebook, and the tracker marks the lesson done. It reads only your imports, functions, classes and constants, so the experiments in your other cells never get in the way.

In [ ]:
check()

## Try it for real

Now run what you wrote on the real thing. Write the steps in the cell at the end; they may call a model with your key, and the checks leave them out.

You need a `GROQ_API_KEY` in your `.env`. Step 1 works on Groq's free plan, but Groq runs batches only on its
Developer tier (three listings cost a fraction of a cent). Otherwise, run steps 3 and 4 on OpenAI's batch API, which
takes the same calls: write the lines with `request_line(..., model="gpt-5-nano")` and pass `OpenAI()` to
`submit_batch` and `collect`. A batch can take several minutes.

1. Load your keys with `load_dotenv(override=True)`. Write three messy listings of your own, then print
   `rewrite(completion, ...)` for the first one, with its cost in cents.
2. Optional, the lesson's free route: with Ollama running, send the same two messages with LiteLLM's `completion`,
   the model `"ollama/llama3.2"` and `api_base="http://localhost:11434"`.
3. Make the batch client: `Groq(api_key=os.environ["GROQ_API_KEY"])` on the Developer tier, otherwise `OpenAI()`.
   Write one `request_line(i, text)` per listing into a `.jsonl` file in a temporary folder, each on its own line
   (with `model="gpt-5-nano"` for OpenAI), and send it with `submit_batch(client, path)`.
4. Every 30 seconds, call `collect` until it gives the summaries, then print them.

<details><summary>Stuck? One way to write it</summary>

```python
import os
import tempfile
import time
from pathlib import Path

from dotenv import load_dotenv
from groq import Groq
from litellm import completion
from openai import OpenAI

load_dotenv(override=True)
LISTINGS = ["WM III PROGRAM!!! 1987 silverdome, light creases on cover, lot #A77",
            "royal rumble 1992 arena poster, rolled, small tear top left, NO RESERVE",
            "1985 wrestling figure still on card!!! soft corners, bubble intact"]

summary, cost = rewrite(completion, LISTINGS[0])
print(summary)
print(f"Cost: {cost * 100:.4f} cents")

# Optional: with Ollama running
# messages = [{"role": "system", "content": SYSTEM_PROMPT}, {"role": "user", "content": LISTINGS[0]}]
# response = completion(model="ollama/llama3.2", messages=messages, api_base="http://localhost:11434")
# print(response.choices[0].message.content)

GROQ_DEVELOPER_TIER = False  # True if your Groq account can run batches
if GROQ_DEVELOPER_TIER:
    client, model = Groq(api_key=os.environ["GROQ_API_KEY"]), "openai/gpt-oss-20b"
else:
    client, model = OpenAI(), "gpt-5-nano"

folder = Path(tempfile.mkdtemp())
with open(folder / "listings.jsonl", "w", encoding="utf-8") as f:
    for i, text in enumerate(LISTINGS):
        f.write(request_line(i, text, model=model) + "\n")

batch_id = submit_batch(client, folder / "listings.jsonl")
results = None
for _ in range(60):
    results = collect(client, batch_id, folder / "results.jsonl")
    if results is not None:
        break
    time.sleep(30)
if results is None:
    print("The batch has not finished; its status is", client.batches.retrieve(batch_id).status)
for i, summary in sorted((results or {}).items()):
    print(f"{i}: {summary}\n")
```

</details>

In [ ]:
# Try it for real: write the steps above here, in as many cells as you like. This cell may call a model with your key; the checks leave it out.